In [ ]:
# ============================================================
# REPRODUCIBILITY PATH SETUP
# ============================================================

from pathlib import Path
import os

# Change TRAFFIC_FORECASTING_ROOT only if the project is stored
# somewhere other than the default Colab location.
PROJECT_ROOT = Path(
    os.environ.get(
        "TRAFFIC_FORECASTING_ROOT",
        "/content/traffic-forecasting-failure-robustness"
    )
)

RAW_DATA_ROOT = PROJECT_ROOT / "data" / "raw"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

RAW_DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Raw data     :", RAW_DATA_ROOT)
print("Outputs      :", OUTPUT_ROOT)


> **Reproducibility note.** This public version uses project-relative paths. By default, the project root is `/content/traffic-forecasting-failure-robustness`. To use another location (including Google Drive), set the environment variable `TRAFFIC_FORECASTING_ROOT` before running the notebook. Raw source archives belong in `data/raw/`; generated artifacts are written under `outputs/`.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

print("✅ Libraries imported successfully.")

In [ ]:
DATA_DIR = str(RAW_DATA_ROOT)

print("Data folder:", DATA_DIR)
print("Folder exists:", os.path.exists(DATA_DIR))

In [ ]:
all_files = []

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        full_path = os.path.join(root, file)
        all_files.append(full_path)

print("=" * 70)
print("FILES FOUND")
print("=" * 70)
print(f"Total files found: {len(all_files):,}")

for f in all_files[:100]:
    print(f)

In [ ]:
import zipfile
import os
import pandas as pd

zip_inventory = []

for zip_path in sorted(all_files):
    if zip_path.lower().endswith(".zip"):

        zip_size_mb = os.path.getsize(zip_path) / (1024 ** 2)

        with zipfile.ZipFile(zip_path, "r") as z:
            members = z.infolist()

            for member in members:
                zip_inventory.append({
                    "ZIP_File": os.path.basename(zip_path),
                    "ZIP_Size_MB": round(zip_size_mb, 2),
                    "Internal_File": member.filename,
                    "Internal_Size_MB": round(member.file_size / (1024 ** 2), 2),
                    "Compressed_Size_MB": round(member.compress_size / (1024 ** 2), 2)
                })

inventory_df = pd.DataFrame(zip_inventory)

print("=" * 100)
print("ZIP FILE INVENTORY")
print("=" * 100)

print(f"Number of ZIP files: {inventory_df['ZIP_File'].nunique()}")
print(f"Total internal files: {len(inventory_df)}")

display(inventory_df)

In [ ]:
zip_summary = (
    inventory_df
    .groupby("ZIP_File")
    .agg(
        ZIP_Size_MB=("ZIP_Size_MB", "first"),
        Number_of_Internal_Files=("Internal_File", "count"),
        Total_Uncompressed_MB=("Internal_Size_MB", "sum")
    )
    .reset_index()
)

zip_summary["Total_Uncompressed_MB"] = zip_summary["Total_Uncompressed_MB"].round(2)

print("=" * 80)
print("MONTHLY ZIP SUMMARY")
print("=" * 80)

display(zip_summary)

print("\nTotal compressed size:",
      round(zip_summary["ZIP_Size_MB"].sum(), 2), "MB")

print("Total estimated uncompressed size:",
      round(zip_summary["Total_Uncompressed_MB"].sum(), 2), "MB")

In [ ]:
import re
from datetime import datetime

date_records = []

for _, row in inventory_df.iterrows():

    filename = os.path.basename(row["Internal_File"])

    # Extract YYYYMMDD from names such as VSDATA_20260416.csv
    match = re.search(r'(\d{8})', filename)

    if match:
        date_string = match.group(1)

        try:
            file_date = pd.to_datetime(date_string, format="%Y%m%d")

            date_records.append({
                "ZIP_File": row["ZIP_File"],
                "Internal_File": row["Internal_File"],
                "Date": file_date
            })

        except ValueError:
            pass

dates_df = pd.DataFrame(date_records)

print("=" * 90)
print("DATE COVERAGE CHECK")
print("=" * 90)

print(f"CSV files detected with valid dates: {len(dates_df):,}")
print(f"Unique dates: {dates_df['Date'].nunique():,}")
print(f"Earliest date: {dates_df['Date'].min().date()}")
print(f"Latest date:   {dates_df['Date'].max().date()}")

# --------------------------------------------------
# Duplicate dates
# --------------------------------------------------

duplicate_dates = (
    dates_df[dates_df.duplicated("Date", keep=False)]
    .sort_values("Date")
)

print("\n" + "=" * 90)
print("DUPLICATE DATE CHECK")
print("=" * 90)

if len(duplicate_dates) == 0:
    print("✅ No duplicate dates found.")
else:
    print(f"⚠️ Duplicate date records: {len(duplicate_dates)}")
    display(duplicate_dates)


# --------------------------------------------------
# Missing dates between January 1 and August 31
# --------------------------------------------------

expected_dates = pd.date_range(
    start="2026-01-01",
    end="2026-08-31",
    freq="D"
)

actual_dates = pd.DatetimeIndex(dates_df["Date"].unique())

missing_dates = expected_dates.difference(actual_dates)

print("\n" + "=" * 90)
print("MISSING DATE CHECK")
print("=" * 90)

print(f"Expected calendar days: {len(expected_dates)}")
print(f"Observed unique days:   {len(actual_dates)}")
print(f"Missing calendar days:  {len(missing_dates)}")

if len(missing_dates) == 0:
    print("✅ No missing calendar dates.")
else:
    print("\nMissing dates:")
    for d in missing_dates:
        print("⚠️", d.date())


# --------------------------------------------------
# Monthly coverage
# --------------------------------------------------

dates_df["Month"] = dates_df["Date"].dt.to_period("M")

monthly_coverage = (
    dates_df
    .groupby("Month")
    .agg(
        First_Date=("Date", "min"),
        Last_Date=("Date", "max"),
        Observed_Days=("Date", "nunique")
    )
    .reset_index()
)

monthly_coverage["Expected_Days"] = (
    monthly_coverage["Month"]
    .apply(lambda x: x.days_in_month)
)

monthly_coverage["Missing_Days"] = (
    monthly_coverage["Expected_Days"]
    - monthly_coverage["Observed_Days"]
)

print("\n" + "=" * 90)
print("MONTHLY DATE COVERAGE")
print("=" * 90)

display(monthly_coverage)

In [ ]:
# ============================================================
# CELL 8 — SCHEMA CONSISTENCY CHECK ACROSS ALL 242 CSV FILES
# ============================================================

import zipfile
import pandas as pd
import os
from collections import Counter

schema_records = []

for zip_path in sorted(all_files):

    if not zip_path.lower().endswith(".zip"):
        continue

    with zipfile.ZipFile(zip_path, "r") as z:

        csv_files = [
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ]

        for csv_name in csv_files:

            try:
                with z.open(csv_name) as f:

                    # Read header only — very memory efficient
                    header_df = pd.read_csv(f, nrows=0)

                columns = tuple(header_df.columns.tolist())

                schema_records.append({
                    "ZIP_File": os.path.basename(zip_path),
                    "CSV_File": csv_name,
                    "Number_of_Columns": len(columns),
                    "Columns": columns,
                    "Status": "OK"
                })

            except Exception as e:

                schema_records.append({
                    "ZIP_File": os.path.basename(zip_path),
                    "CSV_File": csv_name,
                    "Number_of_Columns": None,
                    "Columns": None,
                    "Status": f"ERROR: {str(e)}"
                })


schema_df = pd.DataFrame(schema_records)

print("=" * 100)
print("SCHEMA CONSISTENCY CHECK")
print("=" * 100)

print(f"Files checked: {len(schema_df):,}")
print(f"Files read successfully: {(schema_df['Status'] == 'OK').sum():,}")
print(f"Files with errors: {(schema_df['Status'] != 'OK').sum():,}")


# ------------------------------------------------------------
# Number of columns
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("NUMBER OF COLUMNS DISTRIBUTION")
print("=" * 100)

print(
    schema_df["Number_of_Columns"]
    .value_counts(dropna=False)
    .sort_index()
)


# ------------------------------------------------------------
# Unique schemas
# ------------------------------------------------------------

valid_schemas = schema_df[
    schema_df["Status"] == "OK"
]["Columns"]

schema_counts = Counter(valid_schemas)

print("\n" + "=" * 100)
print("UNIQUE SCHEMAS")
print("=" * 100)

print(f"Number of unique column schemas: {len(schema_counts)}")


for i, (schema, count) in enumerate(
    schema_counts.most_common(),
    start=1
):

    print(f"\nSchema {i}")
    print(f"Files using this schema: {count}")
    print("Columns:")

    for j, col in enumerate(schema, start=1):
        print(f"  {j}. {col}")


# ------------------------------------------------------------
# Identify schema mismatches
# ------------------------------------------------------------

if len(schema_counts) == 1:

    print("\n" + "=" * 100)
    print("FINAL SCHEMA RESULT")
    print("=" * 100)

    print("✅ All CSV files have exactly the same column schema.")

else:

    reference_schema = schema_counts.most_common(1)[0][0]

    mismatch_df = schema_df[
        (schema_df["Status"] == "OK") &
        (schema_df["Columns"] != reference_schema)
    ]

    print("\n" + "=" * 100)
    print("SCHEMA MISMATCHES")
    print("=" * 100)

    print(f"⚠️ Files with different schemas: {len(mismatch_df)}")

    display(
        mismatch_df[
            [
                "ZIP_File",
                "CSV_File",
                "Number_of_Columns",
                "Columns"
            ]
        ]
    )


# ------------------------------------------------------------
# Reading errors
# ------------------------------------------------------------

error_df = schema_df[
    schema_df["Status"] != "OK"
]

if len(error_df) > 0:

    print("\n" + "=" * 100)
    print("FILES WITH READING ERRORS")
    print("=" * 100)

    display(
        error_df[
            [
                "ZIP_File",
                "CSV_File",
                "Status"
            ]
        ]
    )

In [ ]:
# ============================================================
# CELL 9 — INSPECT ONE COMPLETE DAILY FILE
# ============================================================

import zipfile
import pandas as pd
import os

# Use the earliest file in the dataset
first_record = dates_df.sort_values("Date").iloc[0]

target_zip_name = first_record["ZIP_File"]
target_csv_name = first_record["Internal_File"]
target_date = first_record["Date"]

target_zip_path = [
    f for f in all_files
    if os.path.basename(f) == target_zip_name
][0]

print("=" * 100)
print("DAILY FILE INSPECTION")
print("=" * 100)

print("Date:", target_date.date())
print("ZIP:", target_zip_name)
print("CSV:", target_csv_name)

with zipfile.ZipFile(target_zip_path, "r") as z:
    with z.open(target_csv_name) as f:
        daily_df = pd.read_csv(f)

print("\n" + "=" * 100)
print("DATASET SHAPE FOR THIS DAY")
print("=" * 100)

print(f"Rows:    {len(daily_df):,}")
print(f"Columns: {daily_df.shape[1]:,}")

print("\n" + "=" * 100)
print("FIRST 5 ROWS")
print("=" * 100)

display(daily_df.head())

print("\n" + "=" * 100)
print("DATA TYPES")
print("=" * 100)

print(daily_df.dtypes.value_counts())

print("\nDetailed dtypes:")
for col, dtype in daily_df.dtypes.items():
    print(f"{col:<20} {dtype}")

In [ ]:
# ============================================================
# KEY COLUMN INSPECTION
# ============================================================

key_cols = [
    "NB_SCATS_SITE",
    "QT_INTERVAL_COUNT",
    "NB_DETECTOR",
    "NM_REGION",
    "CT_RECORDS",
    "QT_VOLUME_24HOUR",
    "CT_ALARM_24HOUR"
]

print("=" * 100)
print("KEY COLUMN SUMMARY")
print("=" * 100)

for col in key_cols:

    print(f"\n--- {col} ---")

    print("Missing:", daily_df[col].isna().sum())
    print("Unique values:", daily_df[col].nunique(dropna=False))

    unique_values = daily_df[col].drop_duplicates().head(20).tolist()

    print("Example values:", unique_values)

In [ ]:
# ============================================================
# INTERVAL COLUMN CHECK
# ============================================================

volume_cols = [f"V{i:02d}" for i in range(96)]

print("=" * 100)
print("INTERVAL COLUMN CHECK")
print("=" * 100)

print("Number of V columns:", len(volume_cols))

missing_v_cols = [
    col for col in volume_cols
    if col not in daily_df.columns
]

if len(missing_v_cols) == 0:
    print("✅ All V00–V95 columns are present.")
else:
    print("⚠️ Missing interval columns:")
    print(missing_v_cols)

print("\nFirst interval column:", volume_cols[0])
print("Last interval column: ", volume_cols[-1])

print("\nSample of interval values:")
display(
    daily_df[
        ["NB_SCATS_SITE", "NB_DETECTOR"] +
        volume_cols[:8] +
        ["QT_VOLUME_24HOUR"]
    ].head(10)
)

In [ ]:
# ============================================================
# CELL 10 — DAILY TOTAL & INTERVAL INTEGRITY CHECK
# ============================================================

volume_cols = [f"V{i:02d}" for i in range(96)]

check_df = daily_df[
    [
        "NB_SCATS_SITE",
        "NB_DETECTOR",
        "CT_RECORDS",
        "QT_VOLUME_24HOUR",
        "CT_ALARM_24HOUR"
    ] + volume_cols
].copy()

# Sum of the 96 interval values
check_df["CALCULATED_VOLUME"] = check_df[volume_cols].sum(axis=1)

# Difference between supplied daily total and calculated total
check_df["VOLUME_DIFFERENCE"] = (
    check_df["QT_VOLUME_24HOUR"]
    - check_df["CALCULATED_VOLUME"]
)

print("=" * 100)
print("DAILY TOTAL CONSISTENCY CHECK — 2026-01-01")
print("=" * 100)

print(f"Total rows: {len(check_df):,}")

print("\nCT_RECORDS distribution:")
print(check_df["CT_RECORDS"].value_counts(dropna=False).sort_index())


# ============================================================
# DAILY TOTAL VS SUM OF V00–V95
# ============================================================

exact_matches = (
    check_df["QT_VOLUME_24HOUR"]
    == check_df["CALCULATED_VOLUME"]
).sum()

mismatches = len(check_df) - exact_matches

print("\n" + "=" * 100)
print("QT_VOLUME_24HOUR VS SUM(V00:V95)")
print("=" * 100)

print(f"Exact matches: {exact_matches:,}")
print(f"Mismatches:    {mismatches:,}")
print(f"Match rate:    {exact_matches / len(check_df) * 100:.4f}%")


# ============================================================
# NEGATIVE VALUES
# ============================================================

print("\n" + "=" * 100)
print("NEGATIVE VALUE CHECK")
print("=" * 100)

negative_daily = check_df[
    check_df["QT_VOLUME_24HOUR"] < 0
]

negative_interval_mask = check_df[volume_cols] < 0

rows_with_negative_intervals = (
    negative_interval_mask.any(axis=1)
)

print(
    f"Rows with negative QT_VOLUME_24HOUR: "
    f"{len(negative_daily):,}"
)

print(
    f"Rows containing ≥1 negative V interval: "
    f"{rows_with_negative_intervals.sum():,}"
)

print(
    f"Total negative interval cells: "
    f"{negative_interval_mask.sum().sum():,}"
)

print("\nNegative daily-total values:")
print(
    check_df.loc[
        check_df["QT_VOLUME_24HOUR"] < 0,
        "QT_VOLUME_24HOUR"
    ].value_counts().sort_index()
)


# ============================================================
# ZERO VALUES
# ============================================================

print("\n" + "=" * 100)
print("ZERO-VOLUME CHECK")
print("=" * 100)

zero_daily = (
    check_df["QT_VOLUME_24HOUR"] == 0
)

all_intervals_zero = (
    check_df[volume_cols] == 0
).all(axis=1)

print(
    f"Rows with QT_VOLUME_24HOUR = 0: "
    f"{zero_daily.sum():,}"
)

print(
    f"Rows with all 96 intervals = 0: "
    f"{all_intervals_zero.sum():,}"
)

print(
    "Daily total = 0 AND all intervals = 0:",
    (zero_daily & all_intervals_zero).sum()
)


# ============================================================
# INVESTIGATE MISMATCHES
# ============================================================

mismatch_df = check_df[
    check_df["VOLUME_DIFFERENCE"] != 0
]

print("\n" + "=" * 100)
print("MISMATCH EXAMPLES")
print("=" * 100)

print(f"Number of mismatching rows: {len(mismatch_df):,}")

display(
    mismatch_df[
        [
            "NB_SCATS_SITE",
            "NB_DETECTOR",
            "QT_VOLUME_24HOUR",
            "CALCULATED_VOLUME",
            "VOLUME_DIFFERENCE",
            "CT_RECORDS",
            "CT_ALARM_24HOUR"
        ]
    ].head(30)
)


# ============================================================
# NEGATIVE DAILY TOTAL EXAMPLES
# ============================================================

print("\n" + "=" * 100)
print("NEGATIVE DAILY TOTAL EXAMPLES")
print("=" * 100)

display(
    negative_daily[
        [
            "NB_SCATS_SITE",
            "NB_DETECTOR",
            "QT_VOLUME_24HOUR",
            "CALCULATED_VOLUME",
            "VOLUME_DIFFERENCE",
            "CT_RECORDS",
            "CT_ALARM_24HOUR"
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# CELL 11 — NEGATIVE INTERVAL & ALARM MECHANISM CHECK
# ============================================================

import numpy as np
import pandas as pd

volume_cols = [f"V{i:02d}" for i in range(96)]

analysis_df = daily_df[
    [
        "NB_SCATS_SITE",
        "NB_DETECTOR",
        "QT_VOLUME_24HOUR",
        "CT_RECORDS",
        "CT_ALARM_24HOUR"
    ] + volume_cols
].copy()

V = analysis_df[volume_cols].to_numpy()

# ------------------------------------------------------------
# Count interval types per row
# ------------------------------------------------------------

analysis_df["NEG_INTERVAL_COUNT"] = (V < 0).sum(axis=1)
analysis_df["ZERO_INTERVAL_COUNT"] = (V == 0).sum(axis=1)
analysis_df["POS_INTERVAL_COUNT"] = (V > 0).sum(axis=1)

analysis_df["RAW_INTERVAL_SUM"] = V.sum(axis=1)

# Sum traffic volumes while excluding negative sentinel values
analysis_df["VALID_INTERVAL_SUM"] = np.where(V >= 0, V, 0).sum(axis=1)

analysis_df["RAW_MATCH"] = (
    analysis_df["QT_VOLUME_24HOUR"]
    == analysis_df["RAW_INTERVAL_SUM"]
)

analysis_df["HAS_NEGATIVE_INTERVAL"] = (
    analysis_df["NEG_INTERVAL_COUNT"] > 0
)


# ============================================================
# 1. WHAT NEGATIVE VALUES EXIST?
# ============================================================

print("=" * 100)
print("NEGATIVE INTERVAL VALUE DISTRIBUTION")
print("=" * 100)

negative_values = V[V < 0]

values, counts = np.unique(
    negative_values,
    return_counts=True
)

for value, count in zip(values, counts):
    print(f"{value}: {count:,}")


# ============================================================
# 2. DOES ALARM COUNT EQUAL NUMBER OF NEGATIVE INTERVALS?
# ============================================================

print("\n" + "=" * 100)
print("CT_ALARM_24HOUR VS NEGATIVE INTERVAL COUNT")
print("=" * 100)

alarm_match = (
    analysis_df["CT_ALARM_24HOUR"]
    == analysis_df["NEG_INTERVAL_COUNT"]
)

print(f"Exact matches: {alarm_match.sum():,}")
print(f"Mismatches:    {(~alarm_match).sum():,}")
print(
    f"Match rate:    "
    f"{alarm_match.mean() * 100:.4f}%"
)


# ============================================================
# 3. MISMATCH RELATIONSHIP WITH NEGATIVE INTERVALS
# ============================================================

print("\n" + "=" * 100)
print("DAILY-TOTAL MISMATCH VS NEGATIVE INTERVALS")
print("=" * 100)

cross = pd.crosstab(
    analysis_df["HAS_NEGATIVE_INTERVAL"],
    analysis_df["RAW_MATCH"],
    margins=True
)

cross.index.name = "Has_Negative_Interval"
cross.columns.name = "Daily_Total_Matches_Raw_Sum"

display(cross)


# ============================================================
# 4. DAILY TOTAL BEHAVIOR BY ALARM STATUS
# ============================================================

print("\n" + "=" * 100)
print("DAILY TOTAL BEHAVIOR BY ALARM STATUS")
print("=" * 100)

analysis_df["ALARM_STATUS"] = np.where(
    analysis_df["CT_ALARM_24HOUR"] == 0,
    "No Alarm",
    "Alarm Present"
)

analysis_df["DAILY_TOTAL_CLASS"] = np.select(
    [
        analysis_df["QT_VOLUME_24HOUR"] < 0,
        analysis_df["QT_VOLUME_24HOUR"] == 0,
        analysis_df["QT_VOLUME_24HOUR"] > 0
    ],
    [
        "Negative",
        "Zero",
        "Positive"
    ],
    default="Other"
)

alarm_daily_table = pd.crosstab(
    analysis_df["ALARM_STATUS"],
    analysis_df["DAILY_TOTAL_CLASS"],
    margins=True
)

display(alarm_daily_table)


# ============================================================
# 5. DISTRIBUTION OF NUMBER OF ALARMS
# ============================================================

print("\n" + "=" * 100)
print("ALARM COUNT DISTRIBUTION")
print("=" * 100)

print(
    analysis_df["CT_ALARM_24HOUR"]
    .value_counts()
    .sort_index()
    .head(100)
)


# ============================================================
# 6. CHECK SPECIAL CASES
# ============================================================

print("\n" + "=" * 100)
print("SPECIAL CASE SUMMARY")
print("=" * 100)

print(
    "Rows with no alarms:",
    f"{(analysis_df['CT_ALARM_24HOUR'] == 0).sum():,}"
)

print(
    "Rows with ≥1 alarm:",
    f"{(analysis_df['CT_ALARM_24HOUR'] > 0).sum():,}"
)

print(
    "Rows with all 96 intervals negative:",
    f"{(analysis_df['NEG_INTERVAL_COUNT'] == 96).sum():,}"
)

print(
    "Rows with all 96 intervals zero:",
    f"{(analysis_df['ZERO_INTERVAL_COUNT'] == 96).sum():,}"
)

print(
    "Rows with alarms AND QT_VOLUME_24HOUR = -1:",
    f"{((analysis_df['CT_ALARM_24HOUR'] > 0) & (analysis_df['QT_VOLUME_24HOUR'] == -1)).sum():,}"
)

print(
    "Rows with alarms AND QT_VOLUME_24HOUR = 0:",
    f"{((analysis_df['CT_ALARM_24HOUR'] > 0) & (analysis_df['QT_VOLUME_24HOUR'] == 0)).sum():,}"
)

print(
    "Rows with alarms AND positive QT_VOLUME_24HOUR:",
    f"{((analysis_df['CT_ALARM_24HOUR'] > 0) & (analysis_df['QT_VOLUME_24HOUR'] > 0)).sum():,}"
)


# ============================================================
# 7. SHOW ALARM EXAMPLES
# ============================================================

print("\n" + "=" * 100)
print("ALARM EXAMPLES")
print("=" * 100)

display(
    analysis_df.loc[
        analysis_df["CT_ALARM_24HOUR"] > 0,
        [
            "NB_SCATS_SITE",
            "NB_DETECTOR",
            "CT_ALARM_24HOUR",
            "NEG_INTERVAL_COUNT",
            "QT_VOLUME_24HOUR",
            "RAW_INTERVAL_SUM",
            "VALID_INTERVAL_SUM"
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# CELL 12 — FULL 242-DAY DATASET QUALITY SCAN
# Memory-safe: one daily file at a time
# ============================================================

import os
import gc
import zipfile
import numpy as np
import pandas as pd

volume_cols = [f"V{i:02d}" for i in range(96)]

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

OUTPUT_DIR = str(OUTPUT_ROOT / "Notebook_1_Inspection_Outputs")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output directory:")
print(OUTPUT_DIR)


# ------------------------------------------------------------
# Map ZIP filenames to their full paths
# ------------------------------------------------------------

zip_path_map = {
    os.path.basename(path): path
    for path in all_files
    if path.lower().endswith(".zip")
}


# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

daily_results = []

missing_counts_total = pd.Series(
    0,
    index=daily_df.columns,
    dtype="int64"
)

scan_errors = []


# ------------------------------------------------------------
# Main scan
# ------------------------------------------------------------

records_sorted = dates_df.sort_values("Date").reset_index(drop=True)

print("\n" + "=" * 100)
print("STARTING FULL DATASET SCAN")
print("=" * 100)

print(f"Daily files to scan: {len(records_sorted):,}\n")


for i, record in records_sorted.iterrows():

    file_date = pd.Timestamp(record["Date"])
    zip_name = record["ZIP_File"]
    csv_name = record["Internal_File"]

    zip_path = zip_path_map[zip_name]

    try:

        # ====================================================
        # READ ONE DAILY CSV
        # ====================================================

        with zipfile.ZipFile(zip_path, "r") as z:
            with z.open(csv_name) as f:
                df = pd.read_csv(f)

        V = df[volume_cols].to_numpy(copy=False)


        # ====================================================
        # BASIC STRUCTURE
        # ====================================================

        n_rows = len(df)

        unique_sites = df["NB_SCATS_SITE"].nunique()

        unique_site_detector = (
            df[
                ["NB_SCATS_SITE", "NB_DETECTOR"]
            ]
            .drop_duplicates()
            .shape[0]
        )

        duplicate_keys = df.duplicated(
            subset=[
                "NB_SCATS_SITE",
                "QT_INTERVAL_COUNT",
                "NB_DETECTOR"
            ],
            keep=False
        ).sum()


        # ====================================================
        # DATE CONSISTENCY
        # ====================================================

        expected_date_string = file_date.strftime("%Y-%m-%d")

        date_mismatch_rows = (
            df["QT_INTERVAL_COUNT"]
            .astype(str)
            .ne(expected_date_string)
            .sum()
        )

        unique_internal_dates = (
            df["QT_INTERVAL_COUNT"]
            .nunique(dropna=False)
        )


        # ====================================================
        # MISSING VALUES
        # ====================================================

        col_missing = df.isna().sum()

        missing_counts_total = (
            missing_counts_total
            .add(col_missing, fill_value=0)
            .astype("int64")
        )

        total_missing_cells = int(col_missing.sum())

        rows_with_any_missing = int(
            df.isna().any(axis=1).sum()
        )


        # ====================================================
        # CT_RECORDS
        # ====================================================

        records_not_96 = int(
            (df["CT_RECORDS"] != 96).sum()
        )


        # ====================================================
        # INTERVAL QUALITY
        # ====================================================

        negative_mask = V < 0
        minus_one_mask = V == -1
        below_minus_one_mask = V < -1
        zero_mask = V == 0

        negative_interval_cells = int(
            negative_mask.sum()
        )

        minus_one_cells = int(
            minus_one_mask.sum()
        )

        below_minus_one_cells = int(
            below_minus_one_mask.sum()
        )

        rows_with_negative_interval = int(
            negative_mask.any(axis=1).sum()
        )

        all_negative_rows = int(
            minus_one_mask.all(axis=1).sum()
        )

        all_zero_rows = int(
            zero_mask.all(axis=1).sum()
        )


        # ====================================================
        # ALARM CONSISTENCY
        # ====================================================

        neg_count_per_row = (
            minus_one_mask.sum(axis=1)
        )

        alarm_values = (
            df["CT_ALARM_24HOUR"]
            .to_numpy()
        )

        alarm_mismatch_rows = int(
            (alarm_values != neg_count_per_row).sum()
        )

        no_alarm_rows = int(
            (alarm_values == 0).sum()
        )

        partial_alarm_rows = int(
            (
                (alarm_values > 0) &
                (alarm_values < 96)
            ).sum()
        )

        full_alarm_rows = int(
            (alarm_values == 96).sum()
        )

        invalid_alarm_count_rows = int(
            (
                (alarm_values < 0) |
                (alarm_values > 96)
            ).sum()
        )


        # ====================================================
        # DAILY TOTAL
        # ====================================================

        daily_total = (
            df["QT_VOLUME_24HOUR"]
            .to_numpy()
        )

        raw_sum = V.sum(axis=1)

        negative_daily_total_rows = int(
            (daily_total < 0).sum()
        )

        minus_one_daily_total_rows = int(
            (daily_total == -1).sum()
        )

        below_minus_one_daily_total_rows = int(
            (daily_total < -1).sum()
        )

        zero_daily_total_rows = int(
            (daily_total == 0).sum()
        )

        positive_daily_total_rows = int(
            (daily_total > 0).sum()
        )


        # ====================================================
        # DAILY TOTAL CONSISTENCY — CLEAN ROWS ONLY
        # ====================================================

        no_alarm_mask = alarm_values == 0

        no_alarm_total_mismatch = int(
            (
                daily_total[no_alarm_mask]
                !=
                raw_sum[no_alarm_mask]
            ).sum()
        )


        # ====================================================
        # SPECIAL ALARM BEHAVIOR
        # ====================================================

        partial_mask = (
            (alarm_values > 0) &
            (alarm_values < 96)
        )

        full_mask = alarm_values == 96

        partial_alarm_daily_minus1 = int(
            (
                partial_mask &
                (daily_total == -1)
            ).sum()
        )

        partial_alarm_daily_zero = int(
            (
                partial_mask &
                (daily_total == 0)
            ).sum()
        )

        partial_alarm_daily_positive = int(
            (
                partial_mask &
                (daily_total > 0)
            ).sum()
        )

        full_alarm_daily_zero = int(
            (
                full_mask &
                (daily_total == 0)
            ).sum()
        )

        full_alarm_daily_nonzero = int(
            (
                full_mask &
                (daily_total != 0)
            ).sum()
        )


        # ====================================================
        # EXTREME VALUES — INITIAL SCREEN
        # ====================================================

        max_interval_volume = int(V.max())

        max_daily_volume = int(
            daily_total.max()
        )


        # ====================================================
        # SAVE DAILY SUMMARY
        # ====================================================

        daily_results.append({

            "Date": file_date,

            "Rows": n_rows,

            "Unique_Sites": unique_sites,

            "Unique_Site_Detectors":
                unique_site_detector,

            "Duplicate_Key_Rows":
                duplicate_keys,

            "Unique_Internal_Dates":
                unique_internal_dates,

            "Date_Mismatch_Rows":
                date_mismatch_rows,

            "Missing_Cells":
                total_missing_cells,

            "Rows_With_Missing":
                rows_with_any_missing,

            "CT_RECORDS_Not_96":
                records_not_96,

            "Negative_Interval_Cells":
                negative_interval_cells,

            "Minus1_Interval_Cells":
                minus_one_cells,

            "Below_Minus1_Interval_Cells":
                below_minus_one_cells,

            "Rows_With_Negative_Interval":
                rows_with_negative_interval,

            "All_96_Minus1_Rows":
                all_negative_rows,

            "All_96_Zero_Rows":
                all_zero_rows,

            "Alarm_Mismatch_Rows":
                alarm_mismatch_rows,

            "No_Alarm_Rows":
                no_alarm_rows,

            "Partial_Alarm_Rows":
                partial_alarm_rows,

            "Full_Alarm_Rows":
                full_alarm_rows,

            "Invalid_Alarm_Count_Rows":
                invalid_alarm_count_rows,

            "Negative_Daily_Total_Rows":
                negative_daily_total_rows,

            "Minus1_Daily_Total_Rows":
                minus_one_daily_total_rows,

            "Below_Minus1_Daily_Total_Rows":
                below_minus_one_daily_total_rows,

            "Zero_Daily_Total_Rows":
                zero_daily_total_rows,

            "Positive_Daily_Total_Rows":
                positive_daily_total_rows,

            "No_Alarm_Total_Mismatch":
                no_alarm_total_mismatch,

            "Partial_Alarm_Daily_Minus1":
                partial_alarm_daily_minus1,

            "Partial_Alarm_Daily_Zero":
                partial_alarm_daily_zero,

            "Partial_Alarm_Daily_Positive":
                partial_alarm_daily_positive,

            "Full_Alarm_Daily_Zero":
                full_alarm_daily_zero,

            "Full_Alarm_Daily_Nonzero":
                full_alarm_daily_nonzero,

            "Max_Interval_Volume":
                max_interval_volume,

            "Max_Daily_Volume":
                max_daily_volume
        })


        # ====================================================
        # PROGRESS
        # ====================================================

        if (
            (i + 1) % 10 == 0
            or
            (i + 1) == len(records_sorted)
        ):

            print(
                f"✅ Scanned "
                f"{i + 1}/{len(records_sorted)} files "
                f"— through {file_date.date()}"
            )


        # ====================================================
        # MEMORY CLEANUP
        # ====================================================

        del df
        del V
        del negative_mask
        del minus_one_mask
        del zero_mask
        del raw_sum

        gc.collect()


    except Exception as e:

        scan_errors.append({
            "Date": file_date,
            "ZIP_File": zip_name,
            "CSV_File": csv_name,
            "Error": str(e)
        })

        print(
            f"❌ ERROR on {file_date.date()}: {e}"
        )


# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

daily_scan_df = pd.DataFrame(daily_results)

daily_scan_df = daily_scan_df.sort_values(
    "Date"
).reset_index(drop=True)


# ============================================================
# SAVE CHECKPOINT FILES
# ============================================================

daily_output_path = os.path.join(
    OUTPUT_DIR,
    "daily_full_inspection_summary.csv"
)

missing_output_path = os.path.join(
    OUTPUT_DIR,
    "missing_values_by_column.csv"
)

daily_scan_df.to_csv(
    daily_output_path,
    index=False
)

missing_counts_total.rename(
    "Missing_Count"
).to_csv(
    missing_output_path
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("FULL DATASET SCAN COMPLETE")
print("=" * 100)

print(
    f"Files successfully scanned: "
    f"{len(daily_scan_df):,}"
)

print(
    f"Files with errors: "
    f"{len(scan_errors):,}"
)

print(
    f"Total rows scanned: "
    f"{daily_scan_df['Rows'].sum():,}"
)

print(
    f"Date range: "
    f"{daily_scan_df['Date'].min().date()} "
    f"to "
    f"{daily_scan_df['Date'].max().date()}"
)


print("\n" + "=" * 100)
print("CRITICAL QUALITY TOTALS")
print("=" * 100)

critical_cols = [

    "Duplicate_Key_Rows",

    "Date_Mismatch_Rows",

    "Missing_Cells",

    "CT_RECORDS_Not_96",

    "Below_Minus1_Interval_Cells",

    "Alarm_Mismatch_Rows",

    "Invalid_Alarm_Count_Rows",

    "Below_Minus1_Daily_Total_Rows",

    "No_Alarm_Total_Mismatch",

    "Partial_Alarm_Daily_Positive",

    "Full_Alarm_Daily_Nonzero"
]

for col in critical_cols:

    print(
        f"{col:<40} "
        f"{daily_scan_df[col].sum():,}"
    )


print("\n" + "=" * 100)
print("TRAFFIC / ALARM TOTALS")
print("=" * 100)

summary_cols = [

    "No_Alarm_Rows",

    "Partial_Alarm_Rows",

    "Full_Alarm_Rows",

    "Minus1_Interval_Cells",

    "Rows_With_Negative_Interval",

    "All_96_Minus1_Rows",

    "All_96_Zero_Rows",

    "Minus1_Daily_Total_Rows",

    "Zero_Daily_Total_Rows",

    "Positive_Daily_Total_Rows"
]

for col in summary_cols:

    print(
        f"{col:<40} "
        f"{daily_scan_df[col].sum():,}"
    )


print("\nMaximum interval volume:")
print(
    daily_scan_df[
        [
            "Date",
            "Max_Interval_Volume"
        ]
    ]
    .sort_values(
        "Max_Interval_Volume",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)


print("\nMaximum daily volume:")
print(
    daily_scan_df[
        [
            "Date",
            "Max_Daily_Volume"
        ]
    ]
    .sort_values(
        "Max_Daily_Volume",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)


print("\nSaved files:")
print(daily_output_path)
print(missing_output_path)

In [ ]:
# ============================================================
# CELL 13 — TEMPORAL LOCALIZATION OF QUALITY ANOMALIES
# ============================================================

import pandas as pd
import numpy as np

temp = daily_scan_df.copy()

temp["Month"] = temp["Date"].dt.to_period("M")

quality_metrics = [
    "CT_RECORDS_Not_96",
    "Alarm_Mismatch_Rows",
    "No_Alarm_Total_Mismatch"
]

print("=" * 100)
print("DAYS AFFECTED BY EACH QUALITY ISSUE")
print("=" * 100)

for col in quality_metrics:

    affected = temp[temp[col] > 0]

    print(f"\n--- {col} ---")
    print(f"Affected days: {len(affected):,}")

    if len(affected) > 0:
        print(f"First affected date: {affected['Date'].min().date()}")
        print(f"Last affected date:  {affected['Date'].max().date()}")
        print(f"Maximum in one day:  {affected[col].max():,}")


# ============================================================
# MONTHLY TOTALS
# ============================================================

print("\n" + "=" * 100)
print("MONTHLY QUALITY ISSUE TOTALS")
print("=" * 100)

monthly_quality = (
    temp.groupby("Month")
    .agg(
        Rows=("Rows", "sum"),

        CT_RECORDS_Not_96=(
            "CT_RECORDS_Not_96", "sum"
        ),

        Alarm_Mismatch_Rows=(
            "Alarm_Mismatch_Rows", "sum"
        ),

        No_Alarm_Total_Mismatch=(
            "No_Alarm_Total_Mismatch", "sum"
        )
    )
    .reset_index()
)

monthly_quality["CT_RECORDS_Not_96_pct"] = (
    monthly_quality["CT_RECORDS_Not_96"]
    / monthly_quality["Rows"]
    * 100
)

monthly_quality["Alarm_Mismatch_pct"] = (
    monthly_quality["Alarm_Mismatch_Rows"]
    / monthly_quality["Rows"]
    * 100
)

monthly_quality["No_Alarm_Total_Mismatch_pct"] = (
    monthly_quality["No_Alarm_Total_Mismatch"]
    / monthly_quality["Rows"]
    * 100
)

display(monthly_quality)


# ============================================================
# FIRST AFFECTED DATES
# ============================================================

print("\n" + "=" * 100)
print("FIRST 30 AFFECTED DATES")
print("=" * 100)

affected_any = temp[
    (temp["CT_RECORDS_Not_96"] > 0)
    |
    (temp["Alarm_Mismatch_Rows"] > 0)
    |
    (temp["No_Alarm_Total_Mismatch"] > 0)
][
    [
        "Date",
        "Rows",
        "CT_RECORDS_Not_96",
        "Alarm_Mismatch_Rows",
        "No_Alarm_Total_Mismatch",
        "Max_Interval_Volume",
        "Max_Daily_Volume"
    ]
]

display(affected_any.head(30))


# ============================================================
# LAST AFFECTED DATES
# ============================================================

print("\n" + "=" * 100)
print("LAST 30 AFFECTED DATES")
print("=" * 100)

display(affected_any.tail(30))


# ============================================================
# EXTREME DAILY TOTAL VALUES
# ============================================================

print("\n" + "=" * 100)
print("MAX DAILY VOLUME DISTRIBUTION")
print("=" * 100)

print(
    temp["Max_Daily_Volume"]
    .value_counts()
    .sort_index(ascending=False)
    .head(30)
)


print("\nDays where maximum daily volume >= 50,000:")

display(
    temp.loc[
        temp["Max_Daily_Volume"] >= 50000,
        [
            "Date",
            "Max_Daily_Volume",
            "Max_Interval_Volume"
        ]
    ]
)

In [ ]:
# ============================================================
# CELL 14 — TARGETED ANOMALY DIAGNOSIS
# ============================================================

import os
import gc
import zipfile
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

volume_cols = [f"V{i:02d}" for i in range(96)]

records_sorted = dates_df.sort_values("Date").reset_index(drop=True)

# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

ct_records_counter = Counter()

intersection_counts = Counter()

interval_759_cells = 0
interval_gt759_cells = 0

rows_any_759 = 0
rows_all_759 = 0

daily_72864_rows = 0
daily_72864_all759_rows = 0
daily_72864_not_all759_rows = 0

site_detector_72864 = Counter()
site_detector_all759 = Counter()

examples_ct_records = []
examples_alarm_mismatch = []
examples_noalarm_total_mismatch = []
examples_72864 = []
examples_gt759 = []

max_examples = 30


print("=" * 100)
print("STARTING TARGETED ANOMALY DIAGNOSIS")
print("=" * 100)


for i, record in records_sorted.iterrows():

    file_date = pd.Timestamp(record["Date"])
    zip_name = record["ZIP_File"]
    csv_name = record["Internal_File"]

    zip_path = zip_path_map[zip_name]

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(csv_name) as f:
            df = pd.read_csv(f)

    V = df[volume_cols].to_numpy(copy=False)

    ct_records = df["CT_RECORDS"].to_numpy()
    alarms = df["CT_ALARM_24HOUR"].to_numpy()
    daily_total = df["QT_VOLUME_24HOUR"].to_numpy()

    minus1_count = (V == -1).sum(axis=1)
    raw_sum = V.sum(axis=1)

    # ========================================================
    # CT_RECORDS DISTRIBUTION
    # ========================================================

    values, counts = np.unique(
        ct_records,
        return_counts=True
    )

    for value, count in zip(values, counts):
        ct_records_counter[int(value)] += int(count)


    # ========================================================
    # FLAGS
    # ========================================================

    bad_records = ct_records != 96

    alarm_mismatch = alarms != minus1_count

    no_alarm = alarms == 0

    total_mismatch = daily_total != raw_sum

    noalarm_total_mismatch = (
        no_alarm & total_mismatch
    )


    # ========================================================
    # INTERSECTIONS
    # ========================================================

    intersection_counts[
        "CT_RECORDS_Not96"
    ] += int(bad_records.sum())

    intersection_counts[
        "Alarm_Mismatch"
    ] += int(alarm_mismatch.sum())

    intersection_counts[
        "NoAlarm_Total_Mismatch"
    ] += int(noalarm_total_mismatch.sum())

    intersection_counts[
        "BadRecords_AND_AlarmMismatch"
    ] += int(
        (bad_records & alarm_mismatch).sum()
    )

    intersection_counts[
        "BadRecords_AND_NoAlarmTotalMismatch"
    ] += int(
        (bad_records & noalarm_total_mismatch).sum()
    )

    intersection_counts[
        "AlarmMismatch_AND_NoAlarmTotalMismatch"
    ] += int(
        (alarm_mismatch & noalarm_total_mismatch).sum()
    )

    intersection_counts[
        "All_Three"
    ] += int(
        (
            bad_records
            &
            alarm_mismatch
            &
            noalarm_total_mismatch
        ).sum()
    )

    intersection_counts[
        "AlarmMismatch_BUT_CT_RECORDS_96"
    ] += int(
        (
            alarm_mismatch
            &
            (~bad_records)
        ).sum()
    )

    intersection_counts[
        "NoAlarmTotalMismatch_BUT_CT_RECORDS_96"
    ] += int(
        (
            noalarm_total_mismatch
            &
            (~bad_records)
        ).sum()
    )


    # ========================================================
    # 759 ANALYSIS
    # ========================================================

    mask_759 = V == 759
    mask_gt759 = V > 759

    interval_759_cells += int(mask_759.sum())
    interval_gt759_cells += int(mask_gt759.sum())

    any_759 = mask_759.any(axis=1)
    all_759 = mask_759.all(axis=1)

    rows_any_759 += int(any_759.sum())
    rows_all_759 += int(all_759.sum())

    is_72864 = daily_total == 72864

    daily_72864_rows += int(is_72864.sum())

    daily_72864_all759_rows += int(
        (is_72864 & all_759).sum()
    )

    daily_72864_not_all759_rows += int(
        (is_72864 & (~all_759)).sum()
    )


    # ========================================================
    # SITE-DETECTOR RECURRENCE
    # ========================================================

    idx_72864 = np.where(is_72864)[0]

    for idx in idx_72864:

        key = (
            int(df.iloc[idx]["NB_SCATS_SITE"]),
            int(df.iloc[idx]["NB_DETECTOR"])
        )

        site_detector_72864[key] += 1


    idx_all759 = np.where(all_759)[0]

    for idx in idx_all759:

        key = (
            int(df.iloc[idx]["NB_SCATS_SITE"]),
            int(df.iloc[idx]["NB_DETECTOR"])
        )

        site_detector_all759[key] += 1


    # ========================================================
    # EXAMPLE COLLECTOR
    # ========================================================

    def collect_examples(mask, storage, extra_cols=None):

        if len(storage) >= max_examples:
            return

        idxs = np.where(mask)[0]

        for idx in idxs:

            if len(storage) >= max_examples:
                break

            row = {
                "Date": file_date.date(),

                "NB_SCATS_SITE":
                    int(df.iloc[idx]["NB_SCATS_SITE"]),

                "NB_DETECTOR":
                    int(df.iloc[idx]["NB_DETECTOR"]),

                "CT_RECORDS":
                    int(ct_records[idx]),

                "CT_ALARM_24HOUR":
                    int(alarms[idx]),

                "Minus1_Count":
                    int(minus1_count[idx]),

                "QT_VOLUME_24HOUR":
                    int(daily_total[idx]),

                "Raw_Sum":
                    int(raw_sum[idx]),

                "Min_V":
                    int(V[idx].min()),

                "Max_V":
                    int(V[idx].max())
            }

            if extra_cols is not None:
                row.update(extra_cols(idx))

            storage.append(row)


    collect_examples(
        bad_records,
        examples_ct_records
    )

    collect_examples(
        alarm_mismatch,
        examples_alarm_mismatch
    )

    collect_examples(
        noalarm_total_mismatch,
        examples_noalarm_total_mismatch
    )

    collect_examples(
        is_72864,
        examples_72864,
        lambda idx: {
            "All_96_Are_759":
                bool(all_759[idx])
        }
    )

    collect_examples(
        mask_gt759.any(axis=1),
        examples_gt759,
        lambda idx: {
            "Intervals_GT_759":
                int(mask_gt759[idx].sum())
        }
    )


    # ========================================================
    # PROGRESS
    # ========================================================

    if (
        (i + 1) % 20 == 0
        or
        (i + 1) == len(records_sorted)
    ):

        print(
            f"✅ Processed "
            f"{i + 1}/{len(records_sorted)} "
            f"through {file_date.date()}"
        )


    del df, V
    gc.collect()


# ============================================================
# OUTPUT — CT_RECORDS DISTRIBUTION
# ============================================================

print("\n" + "=" * 100)
print("CT_RECORDS VALUE DISTRIBUTION")
print("=" * 100)

ct_records_df = pd.DataFrame(
    sorted(ct_records_counter.items()),
    columns=[
        "CT_RECORDS",
        "Row_Count"
    ]
)

ct_records_df["Percent"] = (
    ct_records_df["Row_Count"]
    / ct_records_df["Row_Count"].sum()
    * 100
)

display(ct_records_df)


# ============================================================
# OUTPUT — INTERSECTION ANALYSIS
# ============================================================

print("\n" + "=" * 100)
print("QUALITY-ISSUE INTERSECTIONS")
print("=" * 100)

for key, value in intersection_counts.items():
    print(f"{key:<50} {value:,}")


# ============================================================
# OUTPUT — 759 / 72864
# ============================================================

print("\n" + "=" * 100)
print("759 / 72864 PATTERN CHECK")
print("=" * 100)

print(
    f"Interval cells equal to 759:        "
    f"{interval_759_cells:,}"
)

print(
    f"Interval cells greater than 759:    "
    f"{interval_gt759_cells:,}"
)

print(
    f"Rows containing at least one 759:   "
    f"{rows_any_759:,}"
)

print(
    f"Rows where all 96 values = 759:     "
    f"{rows_all_759:,}"
)

print(
    f"Rows with daily total = 72,864:      "
    f"{daily_72864_rows:,}"
)

print(
    f"72,864 rows AND all V = 759:         "
    f"{daily_72864_all759_rows:,}"
)

print(
    f"72,864 rows NOT all V = 759:         "
    f"{daily_72864_not_all759_rows:,}"
)


# ============================================================
# TOP RECURRING SITE-DETECTORS
# ============================================================

print("\n" + "=" * 100)
print("TOP SITE-DETECTORS WITH DAILY TOTAL = 72,864")
print("=" * 100)

top_72864 = pd.DataFrame(
    [
        {
            "NB_SCATS_SITE": key[0],
            "NB_DETECTOR": key[1],
            "Days_With_72864": count
        }
        for key, count
        in site_detector_72864.most_common(30)
    ]
)

display(top_72864)


print("\n" + "=" * 100)
print("TOP SITE-DETECTORS WITH ALL 96 INTERVALS = 759")
print("=" * 100)

top_all759 = pd.DataFrame(
    [
        {
            "NB_SCATS_SITE": key[0],
            "NB_DETECTOR": key[1],
            "Days_All_759": count
        }
        for key, count
        in site_detector_all759.most_common(30)
    ]
)

display(top_all759)


# ============================================================
# EXAMPLES
# ============================================================

print("\n" + "=" * 100)
print("CT_RECORDS != 96 EXAMPLES")
print("=" * 100)
display(pd.DataFrame(examples_ct_records))


print("\n" + "=" * 100)
print("ALARM MISMATCH EXAMPLES")
print("=" * 100)
display(pd.DataFrame(examples_alarm_mismatch))


print("\n" + "=" * 100)
print("NO-ALARM DAILY TOTAL MISMATCH EXAMPLES")
print("=" * 100)
display(pd.DataFrame(examples_noalarm_total_mismatch))


print("\n" + "=" * 100)
print("DAILY TOTAL = 72,864 EXAMPLES")
print("=" * 100)
display(pd.DataFrame(examples_72864))


print("\n" + "=" * 100)
print("INTERVAL > 759 EXAMPLES")
print("=" * 100)
display(pd.DataFrame(examples_gt759))

In [ ]:
# ============================================================
# CELL 15 — SITE-DETECTOR TEMPORAL CONTINUITY & ZERO PERSISTENCE
# ============================================================

import os
import gc
import zipfile
import numpy as np
import pandas as pd

volume_cols = [f"V{i:02d}" for i in range(96)]

records_sorted = (
    dates_df
    .sort_values("Date")
    .reset_index(drop=True)
)

TOTAL_DAYS = len(records_sorted)

# Key encoding:
# Site * 100 + Detector
# Detector values are checked to ensure they remain < 100.

capacity = 1_000_000


def zeros_int16():
    return np.zeros(capacity, dtype=np.int16)


days_present = zeros_int16()
days_all_zero = zeros_int16()
days_positive_daily = zeros_int16()
days_any_alarm = zeros_int16()
days_full_alarm = zeros_int16()
days_bad_records = zeros_int16()
days_clean = zeros_int16()
days_clean_zero = zeros_int16()
days_clean_positive = zeros_int16()
days_all_759 = zeros_int16()

first_seen = np.full(
    capacity,
    -1,
    dtype=np.int16
)

last_seen = np.full(
    capacity,
    -1,
    dtype=np.int16
)


# ------------------------------------------------------------
# Dynamic capacity expansion
# ------------------------------------------------------------

def ensure_capacity(required_size):

    global capacity

    global days_present
    global days_all_zero
    global days_positive_daily
    global days_any_alarm
    global days_full_alarm
    global days_bad_records
    global days_clean
    global days_clean_zero
    global days_clean_positive
    global days_all_759
    global first_seen
    global last_seen

    if required_size <= capacity:
        return

    old_capacity = capacity

    capacity = int(
        max(
            required_size + 100_000,
            old_capacity * 1.5
        )
    )

    grow = capacity - old_capacity

    arrays_zero = [
        "days_present",
        "days_all_zero",
        "days_positive_daily",
        "days_any_alarm",
        "days_full_alarm",
        "days_bad_records",
        "days_clean",
        "days_clean_zero",
        "days_clean_positive",
        "days_all_759"
    ]

    for name in arrays_zero:

        arr = globals()[name]

        globals()[name] = np.pad(
            arr,
            (0, grow),
            constant_values=0
        )

    first_seen = np.pad(
        first_seen,
        (0, grow),
        constant_values=-1
    )

    last_seen = np.pad(
        last_seen,
        (0, grow),
        constant_values=-1
    )


# ============================================================
# MAIN SCAN
# ============================================================

print("=" * 100)
print("STARTING SITE-DETECTOR TEMPORAL PROFILE")
print("=" * 100)


for day_idx, record in records_sorted.iterrows():

    file_date = pd.Timestamp(record["Date"])

    zip_path = zip_path_map[
        record["ZIP_File"]
    ]

    csv_name = record["Internal_File"]

    with zipfile.ZipFile(zip_path, "r") as z:

        with z.open(csv_name) as f:

            df = pd.read_csv(
                f,
                usecols=[
                    "NB_SCATS_SITE",
                    "NB_DETECTOR",
                    "CT_RECORDS",
                    "QT_VOLUME_24HOUR",
                    "CT_ALARM_24HOUR"
                ] + volume_cols
            )


    site = df[
        "NB_SCATS_SITE"
    ].to_numpy(dtype=np.int64)

    detector = df[
        "NB_DETECTOR"
    ].to_numpy(dtype=np.int64)


    # Safety check
    if (
        detector.min() < 0
        or
        detector.max() >= 100
    ):

        raise ValueError(
            "Detector IDs outside expected "
            "0–99 range. Key encoding must be changed."
        )


    keys = site * 100 + detector

    ensure_capacity(
        int(keys.max()) + 1
    )


    V = df[
        volume_cols
    ].to_numpy(copy=False)

    ct_records = df[
        "CT_RECORDS"
    ].to_numpy()

    alarm = df[
        "CT_ALARM_24HOUR"
    ].to_numpy()

    daily_total = df[
        "QT_VOLUME_24HOUR"
    ].to_numpy()


    # --------------------------------------------------------
    # ROW STATUS
    # --------------------------------------------------------

    all_zero = (
        V == 0
    ).all(axis=1)

    all_759 = (
        V == 759
    ).all(axis=1)

    positive_daily = (
        daily_total > 0
    )

    any_alarm = (
        alarm > 0
    )

    full_alarm = (
        alarm == 96
    )

    bad_records = (
        ct_records != 96
    )

    clean = (
        (ct_records == 96)
        &
        (alarm == 0)
    )

    clean_zero = (
        clean
        &
        all_zero
    )

    clean_positive = (
        clean
        &
        positive_daily
    )


    # --------------------------------------------------------
    # UPDATE PROFILE ARRAYS
    # --------------------------------------------------------

    days_present[keys] += 1

    days_all_zero[
        keys[all_zero]
    ] += 1

    days_positive_daily[
        keys[positive_daily]
    ] += 1

    days_any_alarm[
        keys[any_alarm]
    ] += 1

    days_full_alarm[
        keys[full_alarm]
    ] += 1

    days_bad_records[
        keys[bad_records]
    ] += 1

    days_clean[
        keys[clean]
    ] += 1

    days_clean_zero[
        keys[clean_zero]
    ] += 1

    days_clean_positive[
        keys[clean_positive]
    ] += 1

    days_all_759[
        keys[all_759]
    ] += 1


    # --------------------------------------------------------
    # FIRST / LAST OBSERVATION
    # --------------------------------------------------------

    new_keys_mask = (
        first_seen[keys] == -1
    )

    first_seen[
        keys[new_keys_mask]
    ] = day_idx

    last_seen[keys] = day_idx


    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if (
        (day_idx + 1) % 20 == 0
        or
        day_idx + 1 == TOTAL_DAYS
    ):

        print(
            f"✅ Processed "
            f"{day_idx + 1}/{TOTAL_DAYS} "
            f"through {file_date.date()}"
        )


    del df, V
    gc.collect()


# ============================================================
# BUILD SITE-DETECTOR PROFILE
# ============================================================

observed_keys = np.flatnonzero(
    days_present > 0
)

sites = (
    observed_keys // 100
)

detectors = (
    observed_keys % 100
)

date_array = (
    records_sorted["Date"]
    .to_numpy()
)

first_dates = date_array[
    first_seen[observed_keys]
]

last_dates = date_array[
    last_seen[observed_keys]
]


profile_df = pd.DataFrame({

    "NB_SCATS_SITE":
        sites,

    "NB_DETECTOR":
        detectors,

    "Days_Present":
        days_present[observed_keys],

    "First_Date":
        first_dates,

    "Last_Date":
        last_dates,

    "Days_All_Zero":
        days_all_zero[observed_keys],

    "Days_Positive_Daily":
        days_positive_daily[observed_keys],

    "Days_Any_Alarm":
        days_any_alarm[observed_keys],

    "Days_Full_Alarm":
        days_full_alarm[observed_keys],

    "Days_CT_RECORDS_Not96":
        days_bad_records[observed_keys],

    "Days_Clean":
        days_clean[observed_keys],

    "Days_Clean_Zero":
        days_clean_zero[observed_keys],

    "Days_Clean_Positive":
        days_clean_positive[observed_keys],

    "Days_All_759":
        days_all_759[observed_keys]
})


# ------------------------------------------------------------
# Ratios
# ------------------------------------------------------------

profile_df["Presence_Rate"] = (
    profile_df["Days_Present"]
    / TOTAL_DAYS
)

profile_df["All_Zero_Rate"] = (
    profile_df["Days_All_Zero"]
    / profile_df["Days_Present"]
)

profile_df["Alarm_Day_Rate"] = (
    profile_df["Days_Any_Alarm"]
    / profile_df["Days_Present"]
)

profile_df["Bad_Record_Day_Rate"] = (
    profile_df["Days_CT_RECORDS_Not96"]
    / profile_df["Days_Present"]
)


# ============================================================
# ZERO-PERSISTENCE CLASSIFICATION
# ============================================================

zero_rate = profile_df["All_Zero_Rate"]

profile_df["Zero_Persistence_Class"] = np.select(

    [
        zero_rate == 0,

        (zero_rate > 0)
        &
        (zero_rate < 0.10),

        (zero_rate >= 0.10)
        &
        (zero_rate < 0.50),

        (zero_rate >= 0.50)
        &
        (zero_rate < 0.90),

        (zero_rate >= 0.90)
        &
        (zero_rate < 1.00),

        zero_rate == 1.00
    ],

    [
        "Never all-zero",
        ">0–<10% zero days",
        "10–<50% zero days",
        "50–<90% zero days",
        "90–<100% zero days",
        "100% all-zero"
    ],

    default="Other"
)


# ============================================================
# OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("SITE-DETECTOR CONTINUITY SUMMARY")
print("=" * 100)

print(
    f"Unique Site-Detector pairs: "
    f"{len(profile_df):,}"
)

print(
    f"Unique Sites: "
    f"{profile_df['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Pairs present all {TOTAL_DAYS} days: "
    f"{(profile_df['Days_Present'] == TOTAL_DAYS).sum():,}"
)

print(
    f"Pairs present fewer than {TOTAL_DAYS} days: "
    f"{(profile_df['Days_Present'] < TOTAL_DAYS).sum():,}"
)

print(
    f"Pairs with at least one positive day: "
    f"{(profile_df['Days_Positive_Daily'] > 0).sum():,}"
)

print(
    f"Pairs never having a positive daily total: "
    f"{(profile_df['Days_Positive_Daily'] == 0).sum():,}"
)

print(
    f"Pairs with ≥1 alarm day: "
    f"{(profile_df['Days_Any_Alarm'] > 0).sum():,}"
)

print(
    f"Pairs with ≥1 CT_RECORDS != 96 day: "
    f"{(profile_df['Days_CT_RECORDS_Not96'] > 0).sum():,}"
)


# ============================================================
# DAYS-PRESENT DISTRIBUTION
# ============================================================

print("\n" + "=" * 100)
print("DAYS PRESENT DISTRIBUTION — MOST COMMON")
print("=" * 100)

print(
    profile_df["Days_Present"]
    .value_counts()
    .sort_values(
        ascending=False
    )
    .head(30)
)


# ============================================================
# ZERO PERSISTENCE
# ============================================================

print("\n" + "=" * 100)
print("ZERO-PERSISTENCE CLASSIFICATION")
print("=" * 100)

zero_summary = (
    profile_df[
        "Zero_Persistence_Class"
    ]
    .value_counts()
    .rename_axis(
        "Class"
    )
    .reset_index(
        name="Site_Detector_Count"
    )
)

zero_summary["Percent"] = (
    zero_summary[
        "Site_Detector_Count"
    ]
    / len(profile_df)
    * 100
)

display(zero_summary)


# ============================================================
# PERSISTENT ZERO DETECTORS
# ============================================================

persistent_zero = profile_df[
    profile_df[
        "Zero_Persistence_Class"
    ] == "100% all-zero"
].copy()

print("\n" + "=" * 100)
print("PERSISTENT ALL-ZERO DETECTORS")
print("=" * 100)

print(
    f"Number of persistent all-zero "
    f"Site-Detector pairs: "
    f"{len(persistent_zero):,}"
)

print(
    f"Of these, present all {TOTAL_DAYS} days: "
    f"{(persistent_zero['Days_Present'] == TOTAL_DAYS).sum():,}"
)

display(
    persistent_zero
    .sort_values(
        [
            "Days_Present",
            "NB_SCATS_SITE",
            "NB_DETECTOR"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
    .head(30)
)


# ============================================================
# ALL-759 DETECTORS
# ============================================================

print("\n" + "=" * 100)
print("SITE-DETECTORS WITH ALL-759 DAYS")
print("=" * 100)

display(
    profile_df[
        profile_df["Days_All_759"] > 0
    ][
        [
            "NB_SCATS_SITE",
            "NB_DETECTOR",
            "Days_Present",
            "Days_All_759",
            "Days_Positive_Daily",
            "Days_Any_Alarm",
            "Days_CT_RECORDS_Not96"
        ]
    ]
    .sort_values(
        "Days_All_759",
        ascending=False
    )
    .head(30)
)


# ============================================================
# SAVE
# ============================================================

profile_output_path = os.path.join(
    OUTPUT_DIR,
    "site_detector_temporal_profile.csv"
)

profile_df.to_csv(
    profile_output_path,
    index=False
)

print("\nSaved:")
print(profile_output_path)

In [ ]:
# ============================================================
# CELL 16 — FUNCTIONAL DETECTOR ACTIVITY CLASSIFICATION
# ============================================================

import numpy as np
import pandas as pd

activity_df = profile_df.copy()

# ------------------------------------------------------------
# Clean-day zero / positive rates
# ------------------------------------------------------------

activity_df["Clean_Zero_Rate"] = np.where(
    activity_df["Days_Clean"] > 0,
    activity_df["Days_Clean_Zero"] /
    activity_df["Days_Clean"],
    np.nan
)

activity_df["Clean_Positive_Rate"] = np.where(
    activity_df["Days_Clean"] > 0,
    activity_df["Days_Clean_Positive"] /
    activity_df["Days_Clean"],
    np.nan
)


# ============================================================
# CLASSIFICATION
# ============================================================

activity_df["Functional_Class"] = np.select(

    [
        # No usable clean observations
        activity_df["Days_Clean"] == 0,

        # Clean observations exist, but never positive
        (
            (activity_df["Days_Clean"] > 0)
            &
            (activity_df["Days_Clean_Positive"] == 0)
        ),

        # Positive traffic only rarely
        (
            activity_df["Clean_Positive_Rate"] > 0
        )
        &
        (
            activity_df["Clean_Positive_Rate"] < 0.10
        ),

        # Mixed / intermittent detector
        (
            activity_df["Clean_Positive_Rate"] >= 0.10
        )
        &
        (
            activity_df["Clean_Positive_Rate"] < 0.90
        ),

        # Mostly active
        activity_df["Clean_Positive_Rate"] >= 0.90
    ],

    [
        "No clean observations",
        "Functionally inactive",
        "Rarely active (<10%)",
        "Intermittently active (10–<90%)",
        "Mostly active (>=90%)"
    ],

    default="Other"
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 100)
print("FUNCTIONAL DETECTOR CLASSIFICATION")
print("=" * 100)

functional_summary = (
    activity_df["Functional_Class"]
    .value_counts()
    .rename_axis("Functional_Class")
    .reset_index(name="Site_Detector_Count")
)

functional_summary["Percent"] = (
    functional_summary["Site_Detector_Count"]
    / len(activity_df)
    * 100
)

display(functional_summary)


# ============================================================
# CLEAN-DAY AVAILABILITY
# ============================================================

print("\n" + "=" * 100)
print("CLEAN-DAY AVAILABILITY")
print("=" * 100)

print(
    f"Pairs with ≥1 clean day: "
    f"{(activity_df['Days_Clean'] > 0).sum():,}"
)

print(
    f"Pairs with zero clean days: "
    f"{(activity_df['Days_Clean'] == 0).sum():,}"
)

print(
    f"Pairs with ≥1 clean positive day: "
    f"{(activity_df['Days_Clean_Positive'] > 0).sum():,}"
)

print(
    f"Pairs with clean days but NEVER positive: "
    f"{((activity_df['Days_Clean'] > 0) & (activity_df['Days_Clean_Positive'] == 0)).sum():,}"
)


# ============================================================
# FUNCTIONALLY INACTIVE DETECTORS
# ============================================================

inactive_df = activity_df[
    activity_df["Functional_Class"]
    == "Functionally inactive"
].copy()

print("\n" + "=" * 100)
print("FUNCTIONALLY INACTIVE DETECTORS")
print("=" * 100)

print(
    f"Total functionally inactive pairs: "
    f"{len(inactive_df):,}"
)

print(
    f"Inactive pairs present all 242 days: "
    f"{(inactive_df['Days_Present'] == TOTAL_DAYS).sum():,}"
)

print(
    f"Inactive pairs with >=200 clean days: "
    f"{(inactive_df['Days_Clean'] >= 200).sum():,}"
)

display(
    inactive_df[
        [
            "NB_SCATS_SITE",
            "NB_DETECTOR",
            "Days_Present",
            "Days_Clean",
            "Days_Clean_Zero",
            "Days_Clean_Positive",
            "Days_Any_Alarm",
            "Days_CT_RECORDS_Not96",
            "Clean_Zero_Rate"
        ]
    ]
    .sort_values(
        ["Days_Clean", "NB_SCATS_SITE"],
        ascending=[False, True]
    )
    .head(30)
)


# ============================================================
# ACTIVE DETECTOR SUMMARY
# ============================================================

active_df = activity_df[
    activity_df["Days_Clean_Positive"] > 0
].copy()

print("\n" + "=" * 100)
print("DETECTORS WITH VALID POSITIVE TRAFFIC")
print("=" * 100)

print(
    f"Site-Detector pairs with valid positive traffic: "
    f"{len(active_df):,}"
)

print(
    f"Unique sites containing ≥1 valid active detector: "
    f"{active_df['NB_SCATS_SITE'].nunique():,}"
)


# ============================================================
# SITE-LEVEL ACTIVITY
# ============================================================

site_activity = (
    activity_df
    .groupby("NB_SCATS_SITE")
    .agg(
        Total_Detectors=("NB_DETECTOR", "count"),

        Active_Detectors=(
            "Days_Clean_Positive",
            lambda x: (x > 0).sum()
        ),

        Functionally_Inactive_Detectors=(
            "Days_Clean_Positive",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

site_activity["Active_Detector_Rate"] = (
    site_activity["Active_Detectors"]
    / site_activity["Total_Detectors"]
)

print("\n" + "=" * 100)
print("SITE-LEVEL ACTIVE DETECTOR SUMMARY")
print("=" * 100)

print(
    f"Total sites: "
    f"{len(site_activity):,}"
)

print(
    f"Sites with ≥1 active detector: "
    f"{(site_activity['Active_Detectors'] > 0).sum():,}"
)

print(
    f"Sites with ZERO active detectors: "
    f"{(site_activity['Active_Detectors'] == 0).sum():,}"
)

display(
    site_activity
    .sort_values(
        "Active_Detector_Rate"
    )
    .head(30)
)


# ============================================================
# SAVE
# ============================================================

activity_output_path = os.path.join(
    OUTPUT_DIR,
    "detector_functional_activity_profile.csv"
)

site_activity_output_path = os.path.join(
    OUTPUT_DIR,
    "site_functional_activity_profile.csv"
)

activity_df.to_csv(
    activity_output_path,
    index=False
)

site_activity.to_csv(
    site_activity_output_path,
    index=False
)

print("\nSaved:")
print(activity_output_path)
print(site_activity_output_path)

In [ ]:
# ============================================================
# CELL 17 — MEMORY-SAFE CLEAN TRAFFIC OUTLIER INSPECTION
# ============================================================

from collections import Counter

interval_counter = Counter()
daily_counter = Counter()

rows_clean_active = 0
rows_clean_zero = 0
rows_clean_all759 = 0

valid_interval_count = 0
valid_daily_count = 0

extreme_interval_examples = []

MAX_EXAMPLES = 30

print("=" * 100)
print("STARTING MEMORY-SAFE CLEAN-TRAFFIC OUTLIER SCAN")
print("=" * 100)


for i, record in records_sorted.iterrows():

    file_date = pd.Timestamp(record["Date"])

    zip_path = zip_path_map[
        record["ZIP_File"]
    ]

    csv_name = record[
        "Internal_File"
    ]

    # --------------------------------------------------------
    # READ ONE DAY ONLY
    # --------------------------------------------------------

    with zipfile.ZipFile(zip_path, "r") as z:

        with z.open(csv_name) as f:

            df = pd.read_csv(
                f,
                usecols=[
                    "NB_SCATS_SITE",
                    "NB_DETECTOR",
                    "CT_RECORDS",
                    "CT_ALARM_24HOUR",
                    "QT_VOLUME_24HOUR"
                ] + volume_cols
            )

    V = df[
        volume_cols
    ].to_numpy(copy=False)

    records = df[
        "CT_RECORDS"
    ].to_numpy()

    alarms = df[
        "CT_ALARM_24HOUR"
    ].to_numpy()

    daily_total = df[
        "QT_VOLUME_24HOUR"
    ].to_numpy()


    # --------------------------------------------------------
    # CLEAN STATUS
    # --------------------------------------------------------

    clean = (
        (records == 96)
        &
        (alarms == 0)
    )

    clean_positive = (
        clean
        &
        (daily_total > 0)
    )

    clean_zero = (
        clean
        &
        (daily_total == 0)
    )

    all759 = (
        V == 759
    ).all(axis=1)

    clean_all759 = (
        clean
        &
        all759
    )


    rows_clean_active += int(
        clean_positive.sum()
    )

    rows_clean_zero += int(
        clean_zero.sum()
    )

    rows_clean_all759 += int(
        clean_all759.sum()
    )


    # --------------------------------------------------------
    # EXCLUDE ALL-759 FLATLINE ROWS
    # --------------------------------------------------------

    valid_rows = (
        clean_positive
        &
        (~all759)
    )

    if valid_rows.any():

        valid_V = V[
            valid_rows
        ]

        valid_daily = daily_total[
            valid_rows
        ]


        # ====================================================
        # INTERVAL FREQUENCY TABLE
        # ====================================================

        unique_v, counts_v = np.unique(
            valid_V,
            return_counts=True
        )

        for value, count in zip(
            unique_v,
            counts_v
        ):

            interval_counter[
                int(value)
            ] += int(count)


        valid_interval_count += int(
            valid_V.size
        )


        # ====================================================
        # DAILY TOTAL FREQUENCY TABLE
        # ====================================================

        unique_d, counts_d = np.unique(
            valid_daily,
            return_counts=True
        )

        for value, count in zip(
            unique_d,
            counts_d
        ):

            daily_counter[
                int(value)
            ] += int(count)


        valid_daily_count += int(
            len(valid_daily)
        )


        # ====================================================
        # EXTREME EXAMPLES > 759
        # ====================================================

        extreme_rows = (
            valid_V > 759
        ).any(axis=1)

        if (
            extreme_rows.any()
            and
            len(extreme_interval_examples)
            < MAX_EXAMPLES
        ):

            original_indices = np.where(
                valid_rows
            )[0]

            local_indices = np.where(
                extreme_rows
            )[0]

            for local_idx in local_indices:

                if (
                    len(extreme_interval_examples)
                    >= MAX_EXAMPLES
                ):
                    break

                original_idx = original_indices[
                    local_idx
                ]

                row_v = V[
                    original_idx
                ]

                extreme_interval_examples.append({

                    "Date":
                        file_date.date(),

                    "NB_SCATS_SITE":
                        int(
                            df.iloc[
                                original_idx
                            ]["NB_SCATS_SITE"]
                        ),

                    "NB_DETECTOR":
                        int(
                            df.iloc[
                                original_idx
                            ]["NB_DETECTOR"]
                        ),

                    "QT_VOLUME_24HOUR":
                        int(
                            daily_total[
                                original_idx
                            ]
                        ),

                    "Max_Interval":
                        int(
                            row_v.max()
                        ),

                    "Intervals_GT_759":
                        int(
                            (row_v > 759).sum()
                        )
                })


    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if (
        (i + 1) % 30 == 0
        or
        (i + 1) == len(records_sorted)
    ):

        print(
            f"✅ Processed "
            f"{i + 1}/{len(records_sorted)} "
            f"through {file_date.date()}"
        )


    del df
    del V

    gc.collect()


# ============================================================
# EXACT QUANTILE FUNCTION FROM FREQUENCY COUNTS
# ============================================================

def quantile_from_counter(counter, q):

    values = np.array(
        sorted(counter.keys()),
        dtype=np.int64
    )

    counts = np.array(
        [counter[v] for v in values],
        dtype=np.int64
    )

    cumulative = np.cumsum(
        counts
    )

    total = cumulative[-1]

    position = q * (total - 1)

    lower_rank = int(
        np.floor(position)
    )

    upper_rank = int(
        np.ceil(position)
    )

    lower_idx = np.searchsorted(
        cumulative,
        lower_rank + 1
    )

    upper_idx = np.searchsorted(
        cumulative,
        upper_rank + 1
    )

    lower_value = values[
        lower_idx
    ]

    upper_value = values[
        upper_idx
    ]

    if lower_rank == upper_rank:
        return float(lower_value)

    fraction = (
        position
        - lower_rank
    )

    return float(
        lower_value
        +
        fraction
        *
        (
            upper_value
            -
            lower_value
        )
    )


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 100)
print("CLEAN ACTIVE TRAFFIC SUMMARY")
print("=" * 100)

print(
    f"Clean positive rows: "
    f"{rows_clean_active:,}"
)

print(
    f"Clean zero rows:     "
    f"{rows_clean_zero:,}"
)

print(
    f"Clean all-759 rows:  "
    f"{rows_clean_all759:,}"
)

print(
    f"Valid interval observations used: "
    f"{valid_interval_count:,}"
)

print(
    f"Valid daily observations used: "
    f"{valid_daily_count:,}"
)


# ============================================================
# QUANTILES
# ============================================================

quantiles = [
    0,
    0.50,
    0.90,
    0.95,
    0.99,
    0.999,
    0.9999,
    1.0
]


print("\n" + "=" * 100)
print("INTERVAL VOLUME QUANTILES")
print("=" * 100)

for q in quantiles:

    value = quantile_from_counter(
        interval_counter,
        q
    )

    print(
        f"{q:>7.4f} : "
        f"{value:,.2f}"
    )


print("\n" + "=" * 100)
print("DAILY VOLUME QUANTILES")
print("=" * 100)

for q in quantiles:

    value = quantile_from_counter(
        daily_counter,
        q
    )

    print(
        f"{q:>7.4f} : "
        f"{value:,.2f}"
    )


# ============================================================
# EXTREME COUNTS
# ============================================================

print("\n" + "=" * 100)
print("EXTREME VALUE COUNTS")
print("=" * 100)

for threshold in [
    500,
    600,
    700,
    759,
    800,
    1000
]:

    count = sum(
        freq
        for value, freq
        in interval_counter.items()
        if value > threshold
    )

    pct = (
        count
        /
        valid_interval_count
        *
        100
    )

    print(
        f"Intervals > {threshold:<4}: "
        f"{count:,} "
        f"({pct:.8f}%)"
    )


# ============================================================
# EXTREME EXAMPLES
# ============================================================

print("\n" + "=" * 100)
print("CLEAN INTERVALS > 759 — EXAMPLES")
print("=" * 100)

display(
    pd.DataFrame(
        extreme_interval_examples
    )
)

In [ ]:
# ============================================================
# CELL 18 — FINAL NOTEBOOK 1 QUALITY SUMMARY
# ============================================================

import os
import pandas as pd

OUTPUT_DIR = str(OUTPUT_ROOT / "Notebook_1_Inspection_Outputs")

summary_text = """
NOTEBOOK 1 — FULL DATA INSPECTION
=================================

DATA COVERAGE
-------------
Period: 2026-01-01 to 2026-08-30
Consecutive days: 242
Daily CSV files: 242
Total rows: 28,584,865
Unique Site-Detector pairs: 119,160
Unique Sites: 4,738

STRUCTURAL QUALITY
------------------
Duplicate Site-Date-Detector rows: 0
Missing cells: 0
Date mismatches: 0
Schema changes: 0
CSV reading errors: 0

All 242 files contained the same 103-column schema.

INTERVAL STRUCTURE
------------------
Each standard complete row contains:
96 traffic-volume intervals (V00–V95)

CT_RECORDS = 96 for 99.625% of all rows.

Rows with CT_RECORDS != 96:
107,190
(~0.375% of all records)

These incomplete rows explain all observed:
- Alarm-count inconsistencies
- No-alarm daily-total inconsistencies

ALARM / INVALID VALUE MECHANISM
-------------------------------
The only negative interval value observed was -1.

For complete records (CT_RECORDS = 96):
CT_ALARM_24HOUR corresponds to the number of -1 intervals.

Therefore:
-1 should be treated as an invalid/alarmed interval,
not as traffic volume.

FUNCTIONAL DETECTOR ACTIVITY
----------------------------
Site-Detector pairs with valid positive traffic:
48,166

Mostly active detectors (>=90%):
45,604

Functionally inactive detectors:
67,579

Of these inactive detectors:
65,065 had at least 200 clean days
without any positive traffic.

Pairs with no clean observations:
3,415

Sites with >=1 active detector:
4,575

Sites with zero active detectors:
163

TEMPORAL CONTINUITY
-------------------
Site-Detector pairs present all 242 days:
117,160

This represents approximately 98.3% of all
observed Site-Detector pairs.

KNOWN FLATLINE ANOMALY
----------------------
A highly suspicious pattern was identified:

V00 = V01 = ... = V95 = 759
QT_VOLUME_24HOUR = 72,864

Total affected rows:
324

Affected Site-Detector pairs:
4

This flatline pattern should be specifically
flagged during data cleaning.

CLEAN TRAFFIC DISTRIBUTION
--------------------------
Valid interval observations analyzed:
1,014,892,896

Interval quantiles:

Median:   13
90%:      94
95%:      126
99%:      191
99.9%:    293
99.99%:   724

Clean intervals >759:
3

Percentage:
0.00000030%

Therefore, values above 759 should NOT be
automatically removed solely because they exceed 759.

FINAL QUALITY ASSESSMENT
------------------------
The dataset is structurally strong and suitable
for proceeding to systematic cleaning and analysis.

The major data-quality mechanisms are identifiable
and can be handled using explicit quality rules.

Recommended cleaning flags for Notebook 2:

1. CT_RECORDS != 96
2. CT_ALARM_24HOUR > 0
3. Interval value = -1
4. Functionally inactive detector
5. No-clean-observation detector
6. All-759 flatline anomaly
7. Extreme values retained initially but flagged
   for targeted review

No cleaning was performed in Notebook 1.
Notebook 1 was diagnostic only.
"""

print(summary_text)

summary_path = os.path.join(
    OUTPUT_DIR,
    "Notebook_1_Final_Quality_Summary.txt"
)

with open(summary_path, "w") as f:
    f.write(summary_text)

print("\n✅ Final summary saved to:")
print(summary_path)